## Phase 3C — Extract Additional Pore Descriptors

Goal: Identify and extract additional structure-derived pore descriptors from CIF files that may improve CO₂ uptake prediction beyond existing geometry, linker, and metal-node descriptors.

Starting dataset:
data/processed/hmof_phase3_metal_descriptors.csv

Planned descriptor groups:
- Additional pore geometry descriptors
- Framework density / cell descriptors
- Coordination or local environment descriptors, if reliable
- Zeo++ or pymatgen-derived descriptors, depending on tool availability

In [ ]:
import sys
from pathlib import Path

# Add the project root so the shared path configuration is importable.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "paths.py").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import DATA_DIR, MODELS_DIR, RAW_DATA_DIR, RESULTS_DIR, RESULTS_ROOT


In [13]:
import os
from pathlib import Path
import pandas as pd
from pymatgen.core import Structure
from tqdm import tqdm

In [ ]:
# Define the output path for the Phase 3B dataset.
# This file will store the CIF-matched dataset plus metal-node descriptor columns.
# Load the saved Phase 3B dataset as the starting point for Phase 3C.
# low_memory=False prevents pandas from guessing column types in small chunks.
phase3_metal_output_path = DATA_DIR / "hmof_phase3_metal_descriptors.csv"
df = pd.read_csv(phase3_metal_output_path, low_memory=False)
print(df.shape)
print("The Phase 3B dataset was loaded successfully with 25,928 MOFs and 37 columns.")
prin

(25928, 66)
The Phase 3B dataset was loaded successfully with 25,928 MOFs and 37 columns.


In [15]:
# Select one CIF path from the Phase 3B dataset for testing Phase 3C descriptor extraction.
# We test one structure first before applying the method to all MOFs.

test_cif_path = df.loc[0, "cif_path"]

print(test_cif_path)

/home/susan/mof-co2-adsorption/data/raw/hmof_json_local/hMOF-0.cif


In [16]:
# Define a function to extract additional structure-derived descriptors from one CIF file.
# These descriptors come from the CIF unit cell and are different from LCD, PLD, void fraction, and surface area.

def extract_structure_descriptors(cif_path):
    """
    Extract additional structure-derived descriptors from one CIF structure.

    Parameters
    ----------
    cif_path : str or pathlib.Path
        Full path to the CIF file.

    Returns
    -------
    dict
        Dictionary containing unit cell volume, density, number of sites,
        lattice lengths, and lattice angles.
    """
    
    structure = Structure.from_file(cif_path)
    lattice = structure.lattice
    
    return {
        "unit_cell_volume": structure.volume,
        "density": structure.density,
        "num_sites": len(structure),
        "lattice_a": lattice.a,
        "lattice_b": lattice.b,
        "lattice_c": lattice.c,
        "lattice_alpha": lattice.alpha,
        "lattice_beta": lattice.beta,
        "lattice_gamma": lattice.gamma,
    }
print(" The Phase 3C structure descriptor function returned the expected descriptor values for one sample MOF.") 

 The Phase 3C structure descriptor function returned the expected descriptor values for one sample MOF.


In [17]:
# Test the structure descriptor function on one sample CIF.
# This confirms that the function returns the planned Phase 3C descriptors.

# 2. Extract descriptors for your test CIF file
test_structure_descriptors = extract_structure_descriptors(test_cif_path)

# 3. Print confirmation and the extracted values
print("The Phase 3C structure descriptor function returned the expected descriptor values for one sample MOF:")
print(test_structure_descriptors)

The Phase 3C structure descriptor function returned the expected descriptor values for one sample MOF:
{'unit_cell_volume': 2077.2578416484507, 'density': 0.6155145074447357, 'num_sites': 53, 'lattice_a': 12.759401000000002, 'lattice_b': 12.759399, 'lattice_c': 12.759393, 'lattice_alpha': 89.967969, 'lattice_beta': 89.980163, 'lattice_gamma': 90.016641}


In [18]:
# Apply the Phase 3C structure descriptor function to the first 5 CIF files.
# This small test checks that the function works across multiple MOF structures.

test_structure_descriptor_results = []

for cif_path in df["cif_path"].head(5):
    descriptors = extract_structure_descriptors(cif_path)
    test_structure_descriptor_results.append(descriptors)

pd.DataFrame(test_structure_descriptor_results)

,unit_cell_volume,density,num_sites,lattice_a,lattice_b,lattice_c,lattice_alpha,lattice_beta,lattice_gamma
0,2077.257842,0.615515,53,12.759401,12.759399,12.759393,89.967969,89.980163,90.016641
1,2077.257842,1.231029,106,12.759401,12.759399,12.759393,89.967969,89.980163,90.016641
2,2077.258057,1.489894,106,12.759401,12.759400,12.759393,90.030930,90.018575,90.015071
3,2077.258072,0.711524,69,12.759401,12.759400,12.759393,89.969639,89.980663,90.013730
4,1591.808592,1.564747,110,12.759401,11.169416,11.169409,90.030874,90.021365,90.011865


In [8]:
# Apply the Phase 3C structure descriptor function to all CIF files.
# This uses the existing cif_path column, so we do not need to rebuild CIF paths.

structure_descriptor_results = []

for cif_path in df["cif_path"]:
    descriptors = extract_structure_descriptors(cif_path)
    structure_descriptor_results.append(descriptors)

print("Number of descriptor rows:", len(structure_descriptor_results))

/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  return parser.parse_structures(primitive=primitive)[0]
/home/susan/mof-co2-adsorption/.venv/lib/python3.12/site-packages/pymatgen/io/cif.py:1817: UserWarning: Issues encountered while parsing CI

Number of descriptor rows: 25928


In [9]:
# Convert all Phase 3C structure descriptor results into a DataFrame.
# This creates one descriptor row for each MOF.

structure_descriptor_df = pd.DataFrame(structure_descriptor_results)

print(structure_descriptor_df.shape)
structure_descriptor_df.head()

(25928, 9)


,unit_cell_volume,density,num_sites,lattice_a,lattice_b,lattice_c,lattice_alpha,lattice_beta,lattice_gamma
0,2077.257842,0.615515,53,12.759401,12.759399,12.759393,89.967969,89.980163,90.016641
1,2077.257842,1.231029,106,12.759401,12.759399,12.759393,89.967969,89.980163,90.016641
2,2077.258057,1.489894,106,12.759401,12.759400,12.759393,90.030930,90.018575,90.015071
3,2077.258072,0.711524,69,12.759401,12.759400,12.759393,89.969639,89.980663,90.013730
4,1591.808592,1.564747,110,12.759401,11.169416,11.169409,90.030874,90.021365,90.011865


In [10]:
# Confirm that the structure descriptor table has the same number of rows as df.
# This verifies that no MOFs were missed during extraction.

print("Rows in df:", len(df))
print("Rows in structure_descriptor_df:", len(structure_descriptor_df))
print("Match:", len(df) == len(structure_descriptor_df))

Rows in df: 25928
Rows in structure_descriptor_df: 25928
Match: True


In [11]:
# Add Phase 3C structure descriptor columns directly to the existing df.
# This keeps one main dataframe for Phase 3.

structure_cols = structure_descriptor_df.columns.tolist()

df[structure_cols] = structure_descriptor_df[structure_cols]

print(df.shape)
df[["filename"] + structure_cols].head()

(25928, 75)


,filename,unit_cell_volume,density,num_sites,lattice_a,lattice_b,lattice_c,lattice_alpha,lattice_beta,lattice_gamma
0,hMOF-0,2077.257842,0.615515,53,12.759401,12.759399,12.759393,89.967969,89.980163,90.016641
1,hMOF-1,2077.257842,1.231029,106,12.759401,12.759399,12.759393,89.967969,89.980163,90.016641
2,hMOF-10,2077.258057,1.489894,106,12.759401,12.759400,12.759393,90.030930,90.018575,90.015071
3,hMOF-100,2077.258072,0.711524,69,12.759401,12.759400,12.759393,89.969639,89.980663,90.013730
4,hMOF-1000,1591.808592,1.564747,110,12.759401,11.169416,11.169409,90.030874,90.021365,90.011865


In [12]:
# Save the Phase 3C dataset with metal-node and structure-derived descriptors.

phase3_structure_output_path = DATA_DIR / "hmof_phase3_structure_descriptors.csv"

df.to_csv(phase3_structure_output_path, index=False)

print("Saved:", phase3_structure_output_path)
print("Dataset shape:", df.shape)

Saved: /home/susan/mof-co2-adsorption/data/processed/hmof_phase3_structure_descriptors.csv
Dataset shape: (25928, 75)
